# AI Device Health Intelligence System (ADHIS)

## Notebook: 05 - Usage Segmentation (Clustering)
**Objective:** Segment device usage patterns into meaningful groups (heavy users, moderate users, battery-risk).  
**Dataset:** `data/raw/device_telemetry.csv`

### What this notebook covers
- Feature scaling for clustering
- KMeans clustering
- Cluster profiling (mean metrics per cluster)
- Save segmentation results for dashboard + recommendations

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
np.random.seed(42)

df = pd.read_csv("../data/raw/device_telemetry.csv")
df.head()

,device_id,battery_cycles,avg_temp,screen_on_time,fast_charging_count,cpu_usage,battery_health
0,1,152,39.943346,6.350490,61,47.558977,84.217729
1,2,485,28.242771,2.136752,118,42.799247,86.421857
2,3,910,38.351099,8.193086,263,54.995024,59.240169
3,4,320,38.384832,1.973999,238,37.287270,77.761681
4,5,156,23.632519,2.260195,254,36.969037,91.372518


In [2]:
cluster_cols = ["screen_on_time", "cpu_usage", "fast_charging_count", "avg_temp", "battery_cycles"]
X = df[cluster_cols].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [3]:
k = 3  # simple and interpretable
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df["usage_cluster"] = kmeans.fit_predict(X_scaled)

df["usage_cluster"].value_counts()

usage_cluster
0    1875
2    1586
1    1539
Name: count, dtype: int64

In [4]:
profile = df.groupby("usage_cluster")[cluster_cols + ["battery_health"]].mean().round(2)
profile

,screen_on_time,cpu_usage,fast_charging_count,avg_temp,battery_cycles,battery_health
usage_cluster,,,,,,
0,6.00,40.37,143.11,34.92,236.57,85.40
1,5.75,38.79,231.21,34.59,696.82,74.46
2,6.19,40.51,69.16,35.09,713.67,77.93


In [5]:
# Make a copy to label clusters
cluster_labels = {}

for c in profile.index:
    row = profile.loc[c]
    if row["screen_on_time"] > profile["screen_on_time"].median() and row["cpu_usage"] > profile["cpu_usage"].median():
        cluster_labels[c] = "Heavy Users"
    elif row["fast_charging_count"] > profile["fast_charging_count"].median() or row["avg_temp"] > profile["avg_temp"].median():
        cluster_labels[c] = "Battery-Risk"
    else:
        cluster_labels[c] = "Moderate Users"

df["cluster_name"] = df["usage_cluster"].map(cluster_labels)
df["cluster_name"].value_counts()

cluster_name
Moderate Users    1875
Heavy Users       1586
Battery-Risk      1539
Name: count, dtype: int64

In [6]:
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(OUT_DIR / "telemetry_with_clusters.csv", index=False)
print("Saved:", OUT_DIR / "telemetry_with_clusters.csv")

Saved: ../data/processed/telemetry_with_clusters.csv


## Summary
- Segmented devices into 3 behavior clusters
- Created human-readable cluster names (Heavy / Moderate / Battery-Risk)
- Saved results for personalized recommendations and Streamlit dashboard